# Phase 6 — sim-to-real on two live replicas

**What this is.** The check that kv-routing-sim's routing results hold on a real fleet. Two vLLM
**0.30.0** servers — **Qwen/Qwen3-4B, one per Tesla T4** — receive the same agentic traffic three times,
routed by round-robin, session-sticky and radix-affinity. The traffic is H2b's: one shared system
prompt (**K1**) and 64 tenant prompts (**K64**). The simulator predicted every run's cache hit rate
**before this notebook ever ran**; the predictions are embedded below, hashed, and committed to the
repo (`artifacts/results/phase6_prereg/predictions.json`).

**What decides it** (registered in `configs/phase6_sim2real.yaml`):
- **S1** — each run's measured hit rate within **10%** of its prediction;
- **S2** — policies predicted **≥ 0.02** apart come out in the same order. The pair that carries the
  headline is K64: radix-affinity (**0.8436**) above session-sticky (**0.8214**).

Hit rates are cache counts, and cache counts are hardware-independent. Timings are T4 timings (fp16,
TRITON_ATTN) and are labelled so.


**How to run.** Kaggle → *Settings* → Accelerator **GPU T4 ×2**, Internet **on** → *Run All*. About
40 minutes: ~10 to install and start two servers, then six runs of ~3–4 minutes. At the end, download
**`phase6_results.zip`** from the Output panel and bring it back. If a cell stops with ⛔, download the
zip anyway: it holds both server logs.


## 0 — Configuration


In [1]:
# ── Generated from configs/phase6_sim2real.yaml. Do not edit here: edit the config and re-run
#    scripts/phase6_prepare.py, or the embedded predictions stop meaning anything.
MODEL = "Qwen/Qwen3-4B"
VLLM_PINNED = "0.30.0"
BLOCK_SIZE = 16
NUM_GPU_BLOCKS = 1537          # --num-gpu-blocks-override; one is vLLM's null block
MAX_MODEL_LEN = 2048
MAX_NUM_SEQS = 256
MAX_NUM_BATCHED_TOKENS = 8192
N_REPLICAS = 2
ROUTER_SEED = 0
POLICIES = ['round-robin', 'session-sticky', 'radix-affinity']
S1_MAX_RELATIVE_ERROR = 0.1
S2_MIN_GAP = 0.02

# ── Box settings. Safe to change: nothing predicted depends on them.
GPU_MEM_UTIL = 0.85
PORTS = [8000, 8001]          # replica i listens on PORTS[i] and runs on GPU i
STARTUP_TIMEOUT_S = 1800      # the first start downloads ~8 GB of weights and compiles
OUT_DIR = "phase6_results"


## 1 — Environment: two GPUs, one server each


In [2]:
import hashlib
import json
import os
import pathlib
import platform
import random
import re
import shutil
import socket
import subprocess
import sys
import threading
import time
import urllib.error
import urllib.request
from collections import OrderedDict
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone

pathlib.Path(OUT_DIR).mkdir(exist_ok=True)


class Stop(Exception):
    """Halts Run-All with a reason. Download phase6_results.zip anyway."""


def sh(cmd, timeout=60):
    try:
        return subprocess.run(cmd, shell=True, capture_output=True, text=True,
                              timeout=timeout, check=False).stdout.strip()
    except Exception as e:  # an unfamiliar box: any binary may be missing
        return f"<failed: {e}>"


ENV = {
    "date_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "platform": platform.platform(),
    "python": sys.version.split()[0],
    "cpu_count": os.cpu_count(),
    "gpus": sh("nvidia-smi --query-gpu=index,name,memory.total,compute_cap,driver_version "
               "--format=csv,noheader"),
    "kaggle": bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE")),
}
for k, v in ENV.items():
    print(f"{k:10s} {v}")
_n_gpus = len([ln for ln in ENV["gpus"].splitlines() if ln.strip() and "failed" not in ln])
if _n_gpus < N_REPLICAS:
    raise Stop(f"⛔ {_n_gpus} GPU(s) visible, {N_REPLICAS} needed: one server per GPU is the "
               "whole point. Kaggle: Settings → Accelerator → GPU T4 ×2.")
if "T4" not in ENV["gpus"]:
    print("\n⚠️  Not a Tesla T4. Hit rates stay valid (cache counts are hardware-independent), "
          "but every timing must be labelled with the GPU above, not as a T4 figure.")


date_utc   2026-09-28T09:41:41+00:00
platform   Linux-6.12.90+-x86_64-with-glibc2.35
python     3.12.13
cpu_count  4
gpus       0, Tesla T4, 15360 MiB, 7.5, 580.159.04
1, Tesla T4, 15360 MiB, 7.5, 580.159.04
kaggle     True


## 2 — Install vLLM, pinned


In [3]:
# PINNED: the predictions were made for this version, and Phase 0 v2 passed every check on it.
!pip install -q vllm==0.30.0


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.9/314.9 MB 5.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 75.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 81.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.7/322.7 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 84.1 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 72.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.2/43.2 MB 44.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 729.6/729.6 kB 24.8 MB/s eta 0:00:00
   ━━

## 3 — Audit: does the installed engine contain the code the model cites?


In [4]:
import importlib.metadata

import vllm

VLLM_VERSION = vllm.__version__
if VLLM_VERSION != VLLM_PINNED:
    raise Stop(f"⛔ vllm {VLLM_VERSION} is imported, but the predictions were made for "
               f"{VLLM_PINNED}. Restart the kernel and run from the top.")


def _version(pkg):
    try:
        return importlib.metadata.version(pkg)
    except importlib.metadata.PackageNotFoundError:
        return None


ENV["packages"] = {p: _version(p) for p in ("vllm", "torch", "transformers", "triton")}
print("packages:", ENV["packages"])

ROOT = pathlib.Path(vllm.__file__).parent
SOURCES = {}
for p in ROOT.rglob("*.py"):
    try:
        SOURCES[str(p.relative_to(ROOT))] = p.read_text(errors="replace")
    except OSError:
        pass


def found_in(needle):
    return next((f for f, text in SOURCES.items() if needle in text), None)


# (what the simulator relies on, the exact source text in vLLM that implements it)
CITED = [
    ("null block reserved at start-up", "self.null_block = self.free_block_queue.popleft()"),
    ("uncached blocks reused first (prepend)", "prepend_n"),
    ("finished requests freed tail-first", "Free blocks in reverse order"),
    ("lookup capped at prompt_length - 1", "max_cache_hit_length = request.num_tokens - 1"),
    ("queries counted at admission only",
     "Record at admission so unscheduled lookups are not counted"),
    ("eviction event fired before hash check", "on_block_evicted(block)"),
    ("block access history bounded to 4", "deque(maxlen=4)"),
    ("reset also clears block metrics", "self.metrics_collector.reset()"),
    ("async scheduler caches on each output",
     "request, request.num_computed_tokens - request.num_output_placeholders"),
    ("full-prompt admission check, default on", "scheduler_reserve_full_isl: bool = True"),
    ("admission watermark, default 0", "watermark: float = Field(default=0.0"),
]
# (what this notebook sends or reads, the source text that serves it)
USED = [
    ("metric vllm:prefix_cache_queries", "vllm:prefix_cache_queries"),
    ("metric vllm:prefix_cache_hits", "vllm:prefix_cache_hits"),
    ("metric vllm:prompt_tokens_by_source", "vllm:prompt_tokens_by_source"),
    ("metric vllm:num_preemptions", "vllm:num_preemptions"),
    ("metric vllm:kv_block_lifetime_seconds", "vllm:kv_block_lifetime_seconds"),
    ("metric vllm:kv_block_idle_before_evict_seconds", "vllm:kv_block_idle_before_evict_seconds"),
    ("metric vllm:kv_block_reuse_gap_seconds", "vllm:kv_block_reuse_gap_seconds"),
    ("metric vllm:num_requests_running", "vllm:num_requests_running"),
    ("flag --num-gpu-blocks-override", "num_gpu_blocks_override"),
    ("flag --enable-prompt-tokens-details", "enable_prompt_tokens_details"),
    ("request field ignore_eos", "ignore_eos"),
    ("POST /reset_prefix_cache", '"/reset_prefix_cache"'),
    ("GET /server_info", '"/server_info"'),
]
AUDIT = {}
print(f"\nvllm {VLLM_VERSION}: {len(SOURCES):,} source files read")
for group, rows in (("the simulator cites", CITED), ("this notebook uses", USED)):
    print(f"── {group}")
    for what, needle in rows:
        AUDIT[what] = found_in(needle)
        print(f"  {'OK' if AUDIT[what] else '--'}  {what:44s} {AUDIT[what] or 'NOT FOUND'}")
AUDIT["vllm:prefix_cache_hit_rate (must NOT exist)"] = found_in('"vllm:prefix_cache_hit_rate"')
print(f"  {'OK' if not AUDIT['vllm:prefix_cache_hit_rate (must NOT exist)'] else '!!'}  "
      "vllm:prefix_cache_hit_rate does not exist (ADR-008)")

missing_used = [w for w, _ in USED if AUDIT[w] is None]
missing_cited = [w for w, _ in CITED if AUDIT[w] is None]
if missing_used:
    raise Stop(f"⛔ this vLLM lacks what the notebook needs: {missing_used}")
if missing_cited:
    print(f"\n⚠️  not found verbatim: {missing_cited}. Run anyway — the counts will show whether "
          "the behaviour itself changed.")


packages: {'vllm': '0.30.0', 'torch': '2.13.0', 'transformers': '5.17.0', 'triton': '3.7.1'}

vllm 0.30.0: 2,681 source files read
── the simulator cites
  OK  null block reserved at start-up              v1/core/block_pool.py
  OK  uncached blocks reused first (prepend)       v1/core/block_pool.py
  OK  finished requests freed tail-first           v1/core/single_type_kv_cache_manager.py
  OK  lookup capped at prompt_length - 1           v1/core/kv_cache_manager.py
  OK  queries counted at admission only            v1/core/sched/scheduler.py
  OK  eviction event fired before hash check       v1/core/block_pool.py
  OK  block access history bounded to 4            v1/core/kv_cache_metrics.py
  OK  reset also clears block metrics              v1/core/block_pool.py
  OK  async scheduler caches on each output        v1/core/sched/async_scheduler.py
  OK  full-prompt admission check, default on      config/scheduler.py
  OK  admission watermark, default 0               config/scheduler.py
─

## 4 — The traffic and the pre-registered predictions (embedded — do not edit)

The traffic is regenerated here from the generator's settings, and must match the
fingerprint of the trace the simulator predicted on — or the notebook stops.


In [5]:
# ── The traffic, regenerated — a port of kv_routing_sim/traces/generator.py (AgenticWorkload) and
#    traces/hashing.py (chain_hashes). The fingerprints prove it made the exact ids predicted on.
WORKLOADS = json.loads(r"""{"K1": {"n_sessions": 120, "turns_per_session": 4, "session_context_tokens": 150, "user_turn_tokens": 40, "assistant_turn_tokens": 60, "output_tokens": 64, "think_time_s": 6.0, "seed": 0, "prefix_sharing_ratio": 0.8, "arrival_rate": 1.0}, "K64": {"n_sessions": 120, "turns_per_session": 4, "session_context_tokens": 150, "user_turn_tokens": 40, "assistant_turn_tokens": 60, "output_tokens": 64, "think_time_s": 6.0, "seed": 0, "prefix_sharing_ratio": 0.8, "arrival_rate": 1.0, "n_shared_prefixes": 64, "prefix_zipf_s": 1.0}}""")
FINGERPRINTS = json.loads(r"""{"K1": "3d5edaf1a02f180c451f9ffb9d1acadbc847bc3c1c5bdb57f7ba91b59a53e794", "K64": "05a4915f6c3a3ea6546e049f419405b8a2102060f2dc59abd15cdc40e74bbe26"}""")
PRED = json.loads(r"""{"runs": {"K1": {"radix-affinity": {"hit_rate": 0.923039, "preemptions": 0, "prefix_cache_hits": 753200, "prefix_cache_queries": 816000, "requests_per_replica": [0, 480], "ttft_p50_ms": 259.048, "ttft_p95_ms": 481.956}, "round-robin": {"hit_rate": 0.879549, "preemptions": 0, "prefix_cache_hits": 717712, "prefix_cache_queries": 816000, "requests_per_replica": [240, 240], "ttft_p50_ms": 231.39, "ttft_p95_ms": 436.975}, "session-sticky": {"hit_rate": 0.921373, "preemptions": 0, "prefix_cache_hits": 751840, "prefix_cache_queries": 816000, "requests_per_replica": [240, 240], "ttft_p50_ms": 192.192, "ttft_p95_ms": 347.25}}, "K64": {"radix-affinity": {"hit_rate": 0.843647, "preemptions": 0, "prefix_cache_hits": 688416, "prefix_cache_queries": 816000, "requests_per_replica": [212, 268], "ttft_p50_ms": 214.957, "ttft_p95_ms": 1255.83}, "round-robin": {"hit_rate": 0.694431, "preemptions": 34, "prefix_cache_hits": 566656, "prefix_cache_queries": 816000, "requests_per_replica": [240, 240], "ttft_p50_ms": 1707.808, "ttft_p95_ms": 5814.959}, "session-sticky": {"hit_rate": 0.821373, "preemptions": 0, "prefix_cache_hits": 670240, "prefix_cache_queries": 816000, "requests_per_replica": [248, 232], "ttft_p50_ms": 228.416, "ttft_p95_ms": 1550.683}}}}""")
PREDICTIONS_SHA256 = "1a6d1212ff83c86a802c48d67cff45c5fcbf9ba56493ed5405fa0112f70e73b6"

_SHARED_BASE, _PREFIX_STRIDE, _SESSION_BASE, _TURN_BASE = 0, 100_000, 10_000_000, 500_000_000
_MIX_MUL, _MIX_ADD, _MASK64 = 0x9E3779B97F4A7C15, 0x632BE59BD9B4E019, 0xFFFFFFFFFFFFFFFF
_LOW, _HIGH = 1000, 100_000  # ENGINE_SAFE_IDS: ordinary text tokens in Qwen3's vocabulary


def _engine_id(raw):
    return _LOW + ((raw * _MIX_MUL + _MIX_ADD) & _MASK64) % (_HIGH - _LOW)


def generate(w):
    """AgenticWorkload._iter_with_tokens + prefix_assignment, sorted by arrival like generate()."""
    turns, ctx = w["turns_per_session"], w["session_context_tokens"]
    user_n, asst_n, r = w["user_turn_tokens"], w["assistant_turn_tokens"], w["prefix_sharing_ratio"]
    mean_private = ctx + user_n + (user_n + asst_n) * (turns - 1) / 2.0
    shared_len = 0 if r <= 0.0 else max(0, round(r / (1.0 - r) * mean_private))
    k_total = w.get("n_shared_prefixes", 1)
    prefixes = [[_SHARED_BASE + k * _PREFIX_STRIDE + i for i in range(shared_len)]
                for k in range(k_total)]
    if k_total == 1:
        assign = [0] * w["n_sessions"]
    else:
        ks = range(k_total)
        weights = [1.0 / (k + 1) ** w.get("prefix_zipf_s", 1.0) for k in ks]
        assign = random.Random(f"shared-prefix-{w['seed']}").choices(ks, weights=weights,
                                                                     k=w["n_sessions"])
    rng, clock, out = random.Random(w["seed"]), 0.0, []
    for s in range(w["n_sessions"]):
        clock += rng.expovariate(w["arrival_rate"])
        shared = prefixes[assign[s]]
        private = [_SESSION_BASE + s * 100_000 + i for i in range(ctx)]
        history = []
        for t in range(turns):
            base = _TURN_BASE + s * 100_000 + t * 1_000
            user = [base + i for i in range(user_n)]
            out.append({"request_id": f"s{s}-t{t}",
                        "arrival_ts": round(clock + t * w["think_time_s"], 6),
                        "session_id": f"s{s}",
                        "ids": [_engine_id(x) for x in shared + private + history + user]})
            history = history + user + [base + 500 + i for i in range(asst_n)]
    out.sort(key=lambda q: q["arrival_ts"])
    return out


def _h64(data):
    return int.from_bytes(hashlib.blake2b(data, digest_size=8).digest(), "big")


def chain_hashes(ids, block_size=BLOCK_SIZE, salt=0):
    """traces/hashing.py: one hash per FULL block, each chained to its parent."""
    out, parent = [], _h64(salt.to_bytes(8, "big"))
    for i in range(len(ids) // block_size):
        body = b"".join(int(t).to_bytes(8, "big", signed=True)
                        for t in ids[i * block_size:(i + 1) * block_size])
        parent = _h64(parent.to_bytes(8, "big") + body)
        out.append(parent)
    return out


def _canonical(obj):
    return json.dumps(obj, separators=(",", ":"), sort_keys=True).encode()


TRACES = {}
for _name, _w in WORKLOADS.items():
    _tr = generate(_w)
    _fp = hashlib.sha256(_canonical([[q["request_id"], q["arrival_ts"], q["session_id"], q["ids"]]
                                     for q in _tr])).hexdigest()
    if _fp != FINGERPRINTS[_name]:
        raise Stop(f"⛔ {_name}: the regenerated trace does not match its fingerprint — this "
                   "Python's random module or the generator port differs from the one predicted on")
    for _q in _tr:
        _q["blocks"] = chain_hashes(_q["ids"])
    TRACES[_name] = _tr
    print(f"{_name:4s} ✓  {len(_tr)} requests · prompts {min(len(q['ids']) for q in _tr)}–"
          f"{max(len(q['ids']) for q in _tr)} tokens · fingerprint {_fp[:16]}…")
if hashlib.sha256(_canonical(PRED)).hexdigest() != PREDICTIONS_SHA256:
    raise Stop("⛔ the embedded predictions do not match their hash: the data cell was edited")
print(f"predictions ✓ {PREDICTIONS_SHA256[:16]}…\n\nPre-registered hit rates (simulator, 2 replicas):")
for _name, _by in PRED["runs"].items():
    print(f"  {_name:4s} " + "   ".join(f"{p} {v['hit_rate']:.4f}" for p, v in _by.items()))


K1   ✓  480 requests · prompts 1550–1850 tokens · fingerprint 3d5edaf1a02f180c…
K64  ✓  480 requests · prompts 1550–1850 tokens · fingerprint 05a4915f6c3a3ea6…
predictions ✓ 1a6d1212ff83c86a…

Pre-registered hit rates (simulator, 2 replicas):
  K1   radix-affinity 0.9230   round-robin 0.8795   session-sticky 0.9214
  K64  radix-affinity 0.8436   round-robin 0.6944   session-sticky 0.8214


## 5 — Two servers


In [6]:
SERVERS = {}  # replica -> {"proc", "logf", "log_path"}
RESET_MODE = "endpoint"  # falls back to "restart" if the dev endpoint is not served


def engine_argv(i):
    return [
        "vllm", "serve", MODEL, "--port", str(PORTS[i]),
        "--dtype", "half",  # T4 has no bf16
        "--block-size", str(BLOCK_SIZE),
        "--num-gpu-blocks-override", str(NUM_GPU_BLOCKS),
        "--max-model-len", str(MAX_MODEL_LEN),
        "--max-num-seqs", str(MAX_NUM_SEQS),
        "--max-num-batched-tokens", str(MAX_NUM_BATCHED_TOKENS),
        "--gpu-memory-utilization", str(GPU_MEM_UTIL),
        "--enable-prefix-caching",
        "--enable-prompt-tokens-details",
        "--seed", "0",
    ]


def port_in_use(port):
    with socket.socket() as s:
        s.settimeout(1)
        return s.connect_ex(("127.0.0.1", port)) == 0


def start_server(i):
    # ⚠️ Refuse to launch while ANYTHING listens on the port: a stale server would answer /health
    # before ours had time to die of "address already in use" (found by Phase 3's dry-run).
    for _ in range(30):
        if not port_in_use(PORTS[i]):
            break
        time.sleep(1)
    else:
        raise Stop(f"⛔ something already serves on port {PORTS[i]} — a server from an earlier "
                   "run? Run the clean-up cell (or !pkill -f 'vllm serve') and start again.")
    # One physical GPU per replica: complete isolation, so TTFT is attributable (ADR-005).
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(i), VLLM_SERVER_DEV_MODE="1")
    log_path = pathlib.Path(OUT_DIR) / f"server_{i}.log"
    logf = open(log_path, "w")  # noqa: SIM115 - must outlive this cell
    proc = subprocess.Popen(engine_argv(i), stdout=logf, stderr=subprocess.STDOUT, env=env)
    SERVERS[i] = {"proc": proc, "logf": logf, "log_path": str(log_path)}
    print(f"launched replica {i}: GPU {i}, port {PORTS[i]}")


def log_text(i):
    return pathlib.Path(SERVERS[i]["log_path"]).read_text(errors="replace") if i in SERVERS else ""


def http(i, method, path, payload=None, timeout=60):
    data = None if payload is None else json.dumps(payload).encode()
    req = urllib.request.Request(f"http://127.0.0.1:{PORTS[i]}{path}", data=data, method=method,
                                 headers={"Content-Type": "application/json"})
    try:
        with urllib.request.urlopen(req, timeout=timeout) as r:
            return r.status, r.read().decode("utf-8", errors="replace")
    except urllib.error.HTTPError as e:
        return e.code, e.read().decode("utf-8", errors="replace")
    except Exception as e:
        return 0, repr(e)


def wait_ready(i, timeout=STARTUP_TIMEOUT_S, every=30):
    """Ready = OUR process is alive AND the port answers, checked in that order."""
    t0, last = time.time(), 0.0
    while time.time() - t0 < timeout:
        if SERVERS[i]["proc"].poll() is not None:
            print(f"  !! replica {i} EXITED (returncode {SERVERS[i]['proc'].returncode}). Log tail:")
            print(log_text(i)[-3000:])
            return False
        if http(i, "GET", "/health", timeout=5)[0] == 200:
            print(f"  replica {i} ready in {time.time() - t0:.0f}s")
            return True
        if time.time() - t0 - last >= every:
            last = time.time() - t0
            tail = log_text(i).strip().splitlines()
            print(f"    ...{last:.0f}s  {tail[-1][:110] if tail else '(log empty)'}")
        time.sleep(5)
    print(f"  !! replica {i} not ready within {timeout}s")
    return False


def stop_server(i):
    if i not in SERVERS:
        return
    SERVERS[i]["proc"].terminate()
    try:
        SERVERS[i]["proc"].wait(timeout=90)
    except subprocess.TimeoutExpired:
        SERVERS[i]["proc"].kill()
    SERVERS[i]["logf"].close()
    del SERVERS[i]


def stop_all():
    for i in list(SERVERS):
        stop_server(i)


def metric(text, family, **labels):
    """Sum a family over matching labels. None = ABSENT, which is not the same as 0."""
    lab = "".join(f'(?=[^}}]*{k}="{v}")' for k, v in labels.items())
    pat = re.compile(r"^" + re.escape(family) + r"(?:_total)?\{" + lab + r"[^}]*\}\s+(\S+)",
                     re.MULTILINE)
    vals = [float(x) for x in pat.findall(text)]
    return sum(vals) if vals else None


COUNTERS = {  # our name -> (metric family, labels)
    "prefix_cache_queries": ("vllm:prefix_cache_queries", {}),
    "prefix_cache_hits": ("vllm:prefix_cache_hits", {}),
    "local_compute": ("vllm:prompt_tokens_by_source", {"source": "local_compute"}),
    "local_cache_hit": ("vllm:prompt_tokens_by_source", {"source": "local_cache_hit"}),
    "preemptions": ("vllm:num_preemptions", {}),
    "prompt_tokens": ("vllm:prompt_tokens", {}),
    "generation_tokens": ("vllm:generation_tokens", {}),
}


def read_counters(text):
    return {k: metric(text, fam, **lab) for k, (fam, lab) in COUNTERS.items()}


def settled_scrape(i, max_wait=300):
    """Scrape once nothing runs or waits AND the counters stopped moving: stats reach /metrics a
    moment after the tokens they describe, so an early read under-counts."""
    t0, prev = time.time(), None
    while True:
        text = http(i, "GET", "/metrics", timeout=30)[1]
        busy = ((metric(text, "vllm:num_requests_running") or 0)
                + (metric(text, "vllm:num_requests_waiting") or 0))
        now = read_counters(text)
        if busy == 0 and now == prev:
            return text
        if time.time() - t0 > max_wait:
            raise Stop(f"⛔ replica {i} never settled within {max_wait}s (busy={busy})")
        prev = now
        time.sleep(1.0)


def deltas(before, after):
    b, a = read_counters(before), read_counters(after)
    return {k: None if a[k] is None else round(a[k] - (b[k] or 0.0)) for k in COUNTERS}


def fresh_caches():
    """A cold cache on BOTH replicas before every run, by /reset_prefix_cache (a restart if the
    endpoint is not served)."""
    global RESET_MODE
    for i in range(N_REPLICAS):
        if RESET_MODE == "endpoint":
            for _ in range(15):
                code, body = http(i, "POST", "/reset_prefix_cache")
                try:
                    ok = code == 200 and json.loads(body).get("success") is True
                except ValueError:
                    ok = False
                if ok:
                    break
                if code in (404, 405):
                    print("  /reset_prefix_cache is not served: restarting the engines per run")
                    RESET_MODE = "restart"
                    break
                time.sleep(2)  # success=false: a finishing request still holds blocks
            else:
                raise Stop(f"⛔ replica {i} refused to reset its prefix cache 15 times")
        if RESET_MODE == "restart":
            stop_server(i)
            start_server(i)
            if not wait_ready(i):
                raise Stop(f"⛔ replica {i} failed to restart — the log tail is above")


print("server helpers ready")


server helpers ready


In [7]:
CAPACITY = {}
for _i in range(N_REPLICAS):  # one at a time: the first start downloads the weights for both
    start_server(_i)
    if not wait_ready(_i):
        raise Stop(f"⛔ replica {_i} did not start — the log tail is above. Download the zip.")
    _log = log_text(_i)
    _m = re.search(r"KV cache size: ([\d,]+) tokens", _log)
    CAPACITY[_i] = {
        "kv_cache_size_tokens_logged": int(_m.group(1).replace(",", "")) if _m else None,
        "expected_tokens": NUM_GPU_BLOCKS * BLOCK_SIZE,
        "override_logged": f"num_gpu_blocks_override={NUM_GPU_BLOCKS}" in _log,
    }
    print(f"  replica {_i}: {CAPACITY[_i]}")
    if CAPACITY[_i]["kv_cache_size_tokens_logged"] != CAPACITY[_i]["expected_tokens"]:
        raise Stop(f"⛔ replica {_i} logged {CAPACITY[_i]['kv_cache_size_tokens_logged']} tokens "
                   f"of KV cache, not {CAPACITY[_i]['expected_tokens']:,}: every prediction "
                   f"assumes exactly {NUM_GPU_BLOCKS} blocks per replica.")


launched replica 0: GPU 0, port 8000
    ...30s  (log empty)
    ...60s  (APIServer pid=207) INFO 09-28 09:47:17 [kernel.py:416] Final IR op priority after setting platform defaults: 
    ...90s  (EngineCore pid=283) INFO 09-28 09:47:45 [cuda.py:538] Using TRITON_ATTN attention backend out of potential ba
    ...120s  (EngineCore pid=283) INFO 09-28 09:47:45 [cuda.py:538] Using TRITON_ATTN attention backend out of potential ba
    ...150s  (EngineCore pid=283) INFO 09-28 09:47:45 [cuda.py:538] Using TRITON_ATTN attention backend out of potential ba
    ...180s  (EngineCore pid=283) INFO 09-28 09:47:45 [cuda.py:538] Using TRITON_ATTN attention backend out of potential ba
    ...210s  (EngineCore pid=283) INFO 09-28 09:47:45 [cuda.py:538] Using TRITON_ATTN attention backend out of potential ba
    ...240s  (EngineCore pid=283) INFO 09-28 09:47:45 [cuda.py:538] Using TRITON_ATTN attention backend out of potential ba
    ...270s  (EngineCore pid=283) INFO 09-28 09:47:45 [cuda.py:538] Using

## 6 — The routers and the client

The three routers are ports of the simulator's policies — the same algorithm and the
same tie-breaking random stream — so a difference between prediction and measurement
is the engine's, not a different router's.


In [8]:
def stream_one(i, ids, max_tokens):
    """One streamed completion to replica i: send time, first-chunk time, and the usage block."""
    body = {"model": MODEL, "prompt": ids, "max_tokens": max_tokens, "temperature": 0.0,
            "ignore_eos": True, "stream": True, "stream_options": {"include_usage": True}}
    req = urllib.request.Request(f"http://127.0.0.1:{PORTS[i]}/v1/completions",
                                 data=json.dumps(body).encode(),
                                 headers={"Content-Type": "application/json"})
    rec = {"t_send": time.perf_counter(), "t_first": None, "usage": None,
           "finish_reason": None, "error": None}
    try:
        with urllib.request.urlopen(req, timeout=1800) as resp:
            for raw in resp:
                line = raw.strip()
                if not line.startswith(b"data:"):
                    continue
                data = line[5:].strip()
                if data == b"[DONE]":
                    break
                obj = json.loads(data)
                if obj.get("choices"):
                    if rec["t_first"] is None:
                        rec["t_first"] = time.perf_counter()
                    reason = obj["choices"][0].get("finish_reason")
                    if reason:
                        rec["finish_reason"] = reason
                if obj.get("usage"):
                    rec["usage"] = obj["usage"]
    except Exception as e:
        rec["error"] = repr(e)
    rec["t_done"] = time.perf_counter()
    return rec


# ── The routers: ports of kv_routing_sim/policies/*.py, same algorithm, same tie-breaking stream.
#    `out[i]` is replica i's in-flight requests — the simulator's running + queued.
def argmin_random(values, rng):
    lo = min(values)
    cand = [i for i, v in enumerate(values) if v == lo]
    return cand[0] if len(cand) == 1 else rng.choice(cand)


class RoundRobin:
    def __init__(self, n):
        self.n, self.cursor = n, 0

    def choose(self, out, q, rng):
        i, self.cursor = self.cursor, (self.cursor + 1) % self.n
        return i

    def observe(self, i, q):
        pass


class SessionSticky:
    def __init__(self, n, max_sessions=1_000_000):
        self.n, self.assigned, self.max_sessions = n, {}, max_sessions

    def choose(self, out, q, rng):
        sid = q["session_id"]
        if sid is None:
            return argmin_random(out, rng)
        if sid in self.assigned:
            return self.assigned[sid]
        chosen = argmin_random(out, rng)
        if len(self.assigned) < self.max_sessions:
            self.assigned[sid] = chosen
        return chosen

    def observe(self, i, q):
        pass


class RadixAffinity:
    """Its OWN map of block -> replicas it dispatched to (a bitmask), never the true caches."""

    def __init__(self, n, router_map_blocks=500_000, load_penalty=0.0):
        self.n, self.owners, self.cap, self.alpha = n, OrderedDict(), router_map_blocks, load_penalty

    def _match(self, i, blocks):
        bit, k = 1 << i, 0
        for h in blocks:
            if not (self.owners.get(h, 0) & bit):
                break
            k += 1
        return k

    def choose(self, out, q, rng):
        blocks = q["blocks"]
        if not blocks:
            return argmin_random(out, rng)
        first = self.owners.get(blocks[0], 0)
        if first == 0:
            return argmin_random(out, rng)
        scores = [self._match(i, blocks) if first & (1 << i) else 0 for i in range(self.n)]
        best = max(scores)
        if best == 0:
            return argmin_random(out, rng)
        if self.alpha > 0.0:
            adj = [s - self.alpha * out[i] for i, s in enumerate(scores)]
            top = max(adj)
            cand = [i for i, a in enumerate(adj) if a == top]
        else:
            cand = [i for i, s in enumerate(scores) if s == best]
        if len(cand) == 1:
            return cand[0]
        return cand[argmin_random([out[i] for i in cand], rng)]

    def observe(self, i, q):
        bit = 1 << i
        for h in q["blocks"]:
            if h in self.owners:
                self.owners[h] |= bit
                self.owners.move_to_end(h)
            else:
                self.owners[h] = bit
                if len(self.owners) > self.cap:
                    self.owners.popitem(last=False)


ROUTERS = {"round-robin": RoundRobin, "session-sticky": SessionSticky,
           "radix-affinity": RadixAffinity}


def nearest_rank(sorted_vals, q):
    if not sorted_vals:
        return None
    return sorted_vals[max(0, min(len(sorted_vals) - 1, round(q * (len(sorted_vals) - 1))))]


def run_one(name, policy, output_tokens, lead_s=2.0):
    """One (workload, policy): reset both caches, replay the trace open-loop at its arrival times,
    route each request as the simulator did, and read both engines' counters."""
    trace = TRACES[name]
    fresh_caches()
    before = [settled_scrape(i) for i in range(N_REPLICAS)]
    router, rng = ROUTERS[policy](N_REPLICAS), random.Random(ROUTER_SEED)
    lock, inflight, routed = threading.Lock(), [0] * N_REPLICAS, []

    def work(i, q):
        rec = stream_one(i, q["ids"], output_tokens)
        with lock:
            inflight[i] -= 1
        return rec

    t_zero = time.perf_counter() + lead_s
    with ThreadPoolExecutor(max_workers=len(trace)) as pool:
        for q in trace:
            delay = t_zero + q["arrival_ts"] - time.perf_counter()
            if delay > 0:
                time.sleep(delay)
            with lock:
                i = router.choose(list(inflight), q, rng)
                router.observe(i, q)
                inflight[i] += 1
            routed.append((q, i, pool.submit(work, i, q)))
        results = [(q, i, f.result()) for q, i, f in routed]
    after = [settled_scrape(i) for i in range(N_REPLICAS)]
    per_replica = [deltas(b, a) for b, a in zip(before, after)]
    recs = []
    for q, i, rec in results:
        u = rec["usage"] or {}
        cached = (u.get("prompt_tokens_details") or {}).get("cached_tokens")
        recs.append({
            "request_id": q["request_id"], "replica": i,
            "valid": (rec["error"] is None and rec["finish_reason"] == "length"
                      and u.get("completion_tokens") == output_tokens
                      and u.get("prompt_tokens") == len(q["ids"]) and cached is not None),
            "error": rec["error"], "cached_tokens": cached,
            "ttft_ms": None if rec["t_first"] is None else (rec["t_first"] - rec["t_send"]) * 1e3,
            "send_offset_s": rec["t_send"] - t_zero, "scheduled_offset_s": q["arrival_ts"],
        })
    hits = sum(d["prefix_cache_hits"] or 0 for d in per_replica)
    queries = sum(d["prefix_cache_queries"] or 0 for d in per_replica)
    measured = hits / queries if queries else None
    predicted = PRED["runs"][name][policy]["hit_rate"]
    rel = None if measured is None else (measured - predicted) / predicted
    ttfts = sorted(r["ttft_ms"] for r in recs if r["ttft_ms"] is not None)
    run = {
        "workload": name, "policy": policy, "per_replica_counters": per_replica,
        "measured_hit_rate": measured, "predicted_hit_rate": predicted, "relative_error": rel,
        "s1_holds": rel is not None and abs(rel) <= S1_MAX_RELATIVE_ERROR,
        "requests_per_replica": [sum(1 for r in recs if r["replica"] == i)
                                 for i in range(N_REPLICAS)],
        "predicted_requests_per_replica": PRED["runs"][name][policy]["requests_per_replica"],
        "preemptions": sum(d["preemptions"] or 0 for d in per_replica),
        "predicted_preemptions": PRED["runs"][name][policy]["preemptions"],
        "ttft_p50_ms": nearest_rank(ttfts, 0.50), "ttft_p95_ms": nearest_rank(ttfts, 0.95),
        "invalid_requests": [r["request_id"] for r in recs if not r["valid"]][:20],
        "worst_send_lag_ms": max(r["send_offset_s"] - r["scheduled_offset_s"] for r in recs) * 1e3,
        "requests": recs,
    }
    print(f"  {name:4s} {policy:15s} hit {measured:.4f} (predicted {predicted:.4f}, "
          f"{rel:+.1%}) {'✅' if run['s1_holds'] else '⛔'} · requests {run['requests_per_replica']} "
          f"(predicted {run['predicted_requests_per_replica']}) · preemptions {run['preemptions']} "
          f"(predicted {run['predicted_preemptions']}) · p95 TTFT {run['ttft_p95_ms'] or 0:.0f} ms"
          f" · {len(run['invalid_requests'])} invalid · send lag ≤ {run['worst_send_lag_ms']:.0f} ms")
    return run


print("routers and client ready")


routers and client ready


In [9]:
# Throwaway requests on ids that share nothing with the traces (every run below starts from a reset)
# — and a check, before any measured run, that each response reports what the comparison needs.
for _i in range(N_REPLICAS):
    _ids = [1000 + 7 * j + _i for j in range(200)]
    _rec = stream_one(_i, _ids, 8)
    _u = _rec["usage"] or {}
    if (_rec["error"] or _rec["finish_reason"] != "length" or _u.get("completion_tokens") != 8
            or (_u.get("prompt_tokens_details") or {}).get("cached_tokens") is None):
        raise Stop(f"⛔ replica {_i}'s warm-up response is unusable — is ignore_eos honoured, is "
                   f"--enable-prompt-tokens-details in effect? {_rec}")
    _absent = [k for k, v in read_counters(http(_i, "GET", "/metrics")[1]).items() if v is None]
    if _absent:
        raise Stop(f"⛔ replica {_i}'s /metrics lacks counters the comparison reads: {_absent}")
print("warm-up done: both replicas report cached_tokens, exact lengths and every counter")


warm-up done: both replicas report cached_tokens, exact lengths and every counter


## 7 — Six runs: two workloads × three routing policies


In [10]:
RUNS = []
for _name in WORKLOADS:
    for _policy in POLICIES:
        RUNS.append(run_one(_name, _policy, WORKLOADS[_name]["output_tokens"]))


  K1   round-robin     hit 0.8795 (predicted 0.8795, +0.0%) ✅ · requests [240, 240] (predicted [240, 240]) · preemptions 15 (predicted 0) · p95 TTFT 14650 ms · 0 invalid · send lag ≤ 10 ms
  K1   session-sticky  hit 0.9214 (predicted 0.9214, -0.0%) ✅ · requests [236, 244] (predicted [240, 240]) · preemptions 0 (predicted 0) · p95 TTFT 1900 ms · 0 invalid · send lag ≤ 6 ms
  K1   radix-affinity  hit 0.9230 (predicted 0.9230, +0.0%) ✅ · requests [0, 480] (predicted [0, 480]) · preemptions 67 (predicted 0) · p95 TTFT 107230 ms · 0 invalid · send lag ≤ 2 ms
  K64  round-robin     hit 0.6947 (predicted 0.6944, +0.0%) ✅ · requests [240, 240] (predicted [240, 240]) · preemptions 64 (predicted 34) · p95 TTFT 110914 ms · 0 invalid · send lag ≤ 1 ms
  K64  session-sticky  hit 0.8099 (predicted 0.8214, -1.4%) ✅ · requests [220, 260] (predicted [248, 232]) · preemptions 71 (predicted 0) · p95 TTFT 71525 ms · 0 invalid · send lag ≤ 2 ms
  K64  radix-affinity  hit 0.8442 (predicted 0.8436, +0.1%) ✅ 

## 8 — The verdict, and the results file


In [11]:
S2 = {}
for _name in WORKLOADS:
    _m = {r["policy"]: r["measured_hit_rate"] for r in RUNS if r["workload"] == _name}
    _p = {p: PRED["runs"][_name][p]["hit_rate"] for p in POLICIES}
    _pairs = []
    for _a in POLICIES:
        for _b in POLICIES:
            if _a < _b and abs(_p[_a] - _p[_b]) >= S2_MIN_GAP:
                _same = (_m[_a] - _m[_b]) * (_p[_a] - _p[_b]) > 0
                _pairs.append({"pair": [_a, _b], "predicted_gap": round(_p[_a] - _p[_b], 4),
                               "measured_gap": round(_m[_a] - _m[_b], 4), "same_order": _same})
    S2[_name] = {"pairs": _pairs, "holds": all(x["same_order"] for x in _pairs)}
    for x in _pairs:
        print(f"  S2 {_name:4s} {x['pair'][0]:>15s} − {x['pair'][1]:<15s} predicted "
              f"{x['predicted_gap']:+.4f}  measured {x['measured_gap']:+.4f}  "
              f"{'✅' if x['same_order'] else '⛔'}")
S1_ALL = all(r["s1_holds"] for r in RUNS)
VERDICT = {"s1_all_runs": S1_ALL, "s2": S2,
           "phase6_holds": S1_ALL and all(v["holds"] for v in S2.values())}
print(f"\nS1 (every run within {S1_MAX_RELATIVE_ERROR:.0%}): {'HOLDS' if S1_ALL else 'FAILS'}")
print(f"Phase 6: {'HOLDS' if VERDICT['phase6_holds'] else 'FALSIFIED'}")

RESULTS = {
    "notebook": "notebooks/phase6_sim2real.ipynb",
    "fingerprints": FINGERPRINTS,
    "predictions_sha256": PREDICTIONS_SHA256,
    "model": MODEL,
    "engine_argv": [engine_argv(i) for i in range(N_REPLICAS)],
    "environment": ENV,
    "metric_and_source_audit": AUDIT,
    "capacity": CAPACITY,
    "reset_mode": RESET_MODE,
    "verdict": VERDICT,
    "runs": RUNS,
}
pathlib.Path(OUT_DIR, "phase6.json").write_text(json.dumps(RESULTS, indent=1, default=str))
_zip = shutil.make_archive(OUT_DIR, "zip", OUT_DIR)
print(f"\nsaved {OUT_DIR}/ and {pathlib.Path(_zip).name} "
      f"({pathlib.Path(_zip).stat().st_size / 1e6:.1f} MB)")
print("⭐ Download phase6_results.zip and bring it back — then run the clean-up cell.")


  S2 K1       round-robin − session-sticky  predicted -0.0418  measured -0.0418  ✅
  S2 K1    radix-affinity − round-robin     predicted +0.0435  measured +0.0435  ✅
  S2 K64      round-robin − session-sticky  predicted -0.1269  measured -0.1152  ✅
  S2 K64   radix-affinity − round-robin     predicted +0.1492  measured +0.1495  ✅
  S2 K64   radix-affinity − session-sticky  predicted +0.0223  measured +0.0343  ✅

S1 (every run within 10%): HOLDS
Phase 6: HOLDS

saved phase6_results/ and phase6_results.zip (0.1 MB)
⭐ Download phase6_results.zip and bring it back — then run the clean-up cell.


## 9 — Clean-up (run after downloading the zip)


In [12]:
stop_all()
print(sh("nvidia-smi --query-compute-apps=pid,used_memory --format=csv"))


pid, used_gpu_memory [MiB]
